# Retrieval-Augmented Generation (RAG) for InvestIQ


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q -U langchain-text-splitters sentence-transformers huggingface_hub chromadb torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.

# 1. Load the RAG Source Document

For this project, I will use the 10-K filing of:

- IBM {{'SimFinId': 69543}, {'Ticker': 'IBM'}}

This document will be used as the source material for the RAG pipeline.

In [4]:
IBM_PATH = "/content/drive/MyDrive/Data Handling and Machine Learning/RAG/IBM/10-K_2026-02-24_0000051143-26-000010.txt"

In [5]:
with open(IBM_PATH, "r", encoding="utf-8") as file:
    ibm_text = file.read()

print("IBM characters:", len(ibm_text))

print("\nIBM preview:")
print(ibm_text[:500])

IBM characters: 211282

IBM preview:
ibm-20251231
CHX
0000051143
2025
FY
false
http://fasb.org/us-gaap/2025#Revenues
http://fasb.org/us-gaap/2025#Revenues
http://fasb.org/us-gaap/2025#Revenues
P1Y
http://fasb.org/us-gaap/2025#SellingGeneralAndAdministrativeExpense
http://fasb.org/us-gaap/2025#SellingGeneralAndAdministrativeExpense
http://fasb.org/us-gaap/2025#SellingGeneralAndAdministrativeExpense
P1Y
P1Y
http://fasb.org/us-gaap/2025#AccountsPayableCurrent
http://fasb.org/us-gaap/2025#AccountsPayableCurrent
http://fasb.org/us-gaap/


## 1.1 Inspect the Source Document Structure

The source file is readable, but its beginning contains `SEC/XBRL metadata` and `accounting tags`.

Before building the **RAG** pipeline, I will inspect the document to determine whether it also contains the narrative text of the $10-K$ filing.

This is important because **RAG** should retrieve meaningful financial and business information rather than primarily retrieving technical `XBRL metadata`.

In [6]:
# Look for common narrative sections in the document

keywords = [
    "Business",
    "Risk Factors",
    "Management's Discussion and Analysis",
    "Management’s Discussion and Analysis",
    "Financial Statements",
]

print("===== IBM =====")

for keyword in keywords:
    position = ibm_text.find(keyword)

    if position != -1:
        print(f"{keyword}: found at character {position}")
    else:
        print(f"{keyword}: not found")

===== IBM =====
Business: found at character 114718
Risk Factors: found at character 114738
Management's Discussion and Analysis: not found
Management’s Discussion and Analysis: found at character 115064
Financial Statements: found at character 115232


## 1.2 Inspect a Narrative Section

The source file contains both structured `SEC/XBRL` information and `10-K` narrative content.

I will now inspect a small section of the narrative text to determine how clean and readable the source material is before designing the chunking step.

In [7]:
# Inspect text around the Business section

position = ibm_text.find("Business")

print("===== IBM =====")
print(ibm_text[position:position + 3000])

===== IBM =====
Business
1
Item 1A. Risk Factors
3
Item 1B. Unresolved Staff Comments
10
Item 1C. Cybersecurity
10
Item 2. Properties
11
Item 3. Legal Proceedings
11
Item 4. Mine Safety Disclosures
11
PART II
12
Item 5. Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
12
Item 6. [Reserved]
12
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations
12
Item 7A. Quantitative and Qualitative Disclosures About Market Risk
12
Item 8. Financial Statements and Supplementary Data
12
Item 9. Changes in and Disagreements with Accountants on Accounting and Financial Disclosure
12
Item 9A. Controls and Procedures
12
Item 9B. Other Information
13
Item 9C. Disclosure Regarding Foreign Jurisdictions that Prevent Inspections
13
PART III
14
Item 10. Directors, Executive Officers and Corporate Governance
14
Item 11. Executive Compensation
14
Item 12. Security Ownership of Certain Beneficial Owners and Manage

## 1.3 Locate the Main 10-K Narrative

The source file contains both `XBRL metadata`, a `Table of Contents`, and the actual `10-K narrative content`.

The first occurrence of the Business section appears in the Table of Contents, while the actual narrative begins later with `Item 1. Business:`.

I will therefore inspect the formatting and occurrences of the main `Item 1.` heading before defining the section extraction logic.

In [8]:
# Inspect the exact characters around "Item 1."

position = ibm_text.find("Item 1.")

print("First 'Item 1.' found at:", position)

if position != -1:
    sample = ibm_text[position:position + 100]

    print("\nVisible text:")
    print(sample)

    print("\nCharacter representation:")
    print(repr(sample))

First 'Item 1.' found at: 114710

Visible text:
Item 1. Business
1
Item 1A. Risk Factors
3
Item 1B. Unresolved Staff Comments
10
Item 1C. Cybersecur

Character representation:
'Item 1. Business\n1\nItem 1A. Risk Factors\n3\nItem 1B. Unresolved Staff Comments\n10\nItem 1C. Cybersecur'


In [10]:
# Normalize non-breaking spaces

ibm_clean = ibm_text.replace("\xa0", " ")

print("IBM:")
print("Before:", len(ibm_text))
print("After: ", len(ibm_clean))

IBM:
Before: 211282
After:  211282


In [10]:
# Verify that the non-breaking spaces were replaced

print("IBM:")
print("Raw search:  ", ibm_text.find("Item 1."))
print("Clean search:", ibm_clean.find("Item 1."))

IBM:
Raw search:   114710
Clean search: 114710


In [11]:
# Find occurrences of "Item 1." in the document

print("===== IBM =====")

start = 0
count = 0

while True:
    position = ibm_clean.find("Item 1.", start)

    if position == -1:
        break

    print(f"\n--- Occurrence {count + 1} at character {position} ---")
    print(ibm_clean[position:position + 1000])

    start = position + 1
    count += 1

===== IBM =====

--- Occurrence 1 at character 114710 ---
Item 1. Business
1
Item 1A. Risk Factors
3
Item 1B. Unresolved Staff Comments
10
Item 1C. Cybersecurity
10
Item 2. Properties
11
Item 3. Legal Proceedings
11
Item 4. Mine Safety Disclosures
11
PART II
12
Item 5. Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
12
Item 6. [Reserved]
12
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations
12
Item 7A. Quantitative and Qualitative Disclosures About Market Risk
12
Item 8. Financial Statements and Supplementary Data
12
Item 9. Changes in and Disagreements with Accountants on Accounting and Financial Disclosure
12
Item 9A. Controls and Procedures
12
Item 9B. Other Information
13
Item 9C. Disclosure Regarding Foreign Jurisdictions that Prevent Inspections
13
PART III
14
Item 10. Directors, Executive Officers and Corporate Governance
14
Item 11. Executive Compensation
14
Item 12. Security

### 1.3.1 Locate the Main 10-K Section Boundaries

The cleaned document contains multiple sections from the 10-K filing.

Before extracting the narrative content, I will locate the main section headings and their positions in the document.

These positions will help identify the boundaries between sections such as Business, Risk Factors, Management's Discussion and Analysis, and Financial Statements.

In [12]:
section_headers = [
    "Item 1. Business",
    "Item 1A. Risk Factors",
    "Item 7. Management’s Discussion and Analysis",
    "Item 8. Financial Statements",
]

print("========== IBM ==========")

for header in section_headers:
    print(f"\n{header}")

    start = 0

    while True:
        position = ibm_clean.find(header, start)

        if position == -1:
            break

        print("  ", position)

        start = position + 1

========== IBM ==========

Item 1. Business
   114710
   116115

Item 1A. Risk Factors
   114729
   126233

Item 7. Management’s Discussion and Analysis
   115056
   173278

Item 8. Financial Statements
   115224
   173700


---
# 2. Verify the Business Section Boundary

The actual `Item 1. Business` section should end when the actual `Item 1A. Risk Factors` section begins.

I will inspect the text immediately before and after this boundary for IBM to make sure the section extraction will preserve the correct content.

In [13]:
business_header = "Item 1. Business"
risk_header = "Item 1A. Risk Factors"

business_positions = []
start = 0

while True:
    position = ibm_clean.find(business_header, start)

    if position == -1:
        break

    business_positions.append(position)
    start = position + 1


risk_positions = []
start = 0

while True:
    position = ibm_clean.find(risk_header, start)

    if position == -1:
        break

    risk_positions.append(position)
    start = position + 1


# Use the last occurrence of each heading,
# which corresponds to the actual filing narrative.
business_start = business_positions[-1]
risk_start = risk_positions[-1]

print("========== IBM ==========")
print("Business starts at:", business_start)
print("Risk Factors starts at:", risk_start)
print("Business section length:", risk_start - business_start)

print("\n--- End of Business section ---")
print(ibm_clean[risk_start - 500:risk_start + 300])

========== IBM ==========
Business starts at: 116115
Risk Factors starts at: 126233
Business section length: 10118

--- End of Business section ---
r
51
2023
(1)    Member of the Board of Directors.
All executive officers are elected by the Board of Directors annually as provided in the Company’s By-laws. Each executive officer named above, with the exception of Anne Robinson, has been an executive of IBM or its subsidiaries during the past five years. Ms. Robinson previously served as Managing Director, General Counsel and Corporate Secretary of The Vanguard Group, Inc. and Secretary of the Vanguard funds from August 2016 until June 2024.
Item 1A. Risk Factors:
Risks Related to Our Business
Downturn in Economic Environment and Client Spending Budgets Could Impact the Company’s Business:
If overall demand for IBM’s products and solutions decreases, whether due to general economic conditions, or a shift in client buying patterns, the c


## 2.1 Define the RAG Section Boundaries

For the RAG baseline, I will extract four sections from IBM's 10-K filing:

1. Business
2. Risk Factors
3. Management's Discussion and Analysis (MD&A)
4. Financial Statements and related notes

The section boundaries will be determined from the actual narrative occurrences of the 10-K headings rather than fixed character positions.

The complete Financial Statements section will be preserved, including its tables and notes.

In [11]:
section_patterns = {
    "Business": [
        "Item 1. Business"
    ],
    "Risk Factors": [
        "Item 1A. Risk Factors"
    ],
    "MD&A": [
        "Item 7. Management’s Discussion and Analysis"
    ],
    "Financial Statements": [
        "Item 8. Financial Statements"
    ]
}

print("Section patterns defined:")

for section, patterns in section_patterns.items():
    print(f"\n{section}:")
    for pattern in patterns:
        print(repr(pattern))

Section patterns defined:

Business:
'Item 1. Business'

Risk Factors:
'Item 1A. Risk Factors'

MD&A:
'Item 7. Management’s Discussion and Analysis'

Financial Statements:
'Item 8. Financial Statements'


## 2.2 Locate the Actual 10-K Section Headings

Each heading appears more than once because the document also contains a Table of Contents.

The last occurrence of each heading corresponds to the actual 10-K narrative section, so these positions will be used to define the extraction boundaries.

In [12]:
section_positions = {}

for section, patterns in section_patterns.items():
    pattern = patterns[0]

    positions = []
    start = 0

    while True:
        position = ibm_clean.find(pattern, start)

        if position == -1:
            break

        positions.append(position)
        start = position + 1

    section_positions[section] = positions[-1]

print("Actual section positions:")

for section, position in section_positions.items():
    print(f"{section}: {position}")

Actual section positions:
Business: 116115
Risk Factors: 126233
MD&A: 173278
Financial Statements: 173700


## 2.3 Verify the MD&A Section

The identified MD&A and Financial Statements headings are located very close together.

I will inspect this section to determine whether the full MD&A content is included in the 10-K text or incorporated by reference from another part of IBM's annual report.

In [16]:
mdna_start = section_positions["MD&A"]
financial_start = section_positions["Financial Statements"]

print(ibm_clean[mdna_start:financial_start + 500])

Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations:
Refer to pages 6 through 38 of IBM’s 2025 Annual Report to Stockholders, which are incorporated herein by reference.
Item 7A. Quantitative and Qualitative Disclosures About Market Risk:
Refer to the section titled “Market Risk” on page 37 of IBM’s 2025 Annual Report to Stockholders, which is incorporated herein by reference.
Item 8. Financial Statements and Supplementary Data:
Refer to pages 42 through 116 of IBM’s 2025 Annual Report to Stockholders, which are incorporated herein by reference. Also refer to the Financial Statement Schedule on page S-1 of this Form 10-K.
Item 9. Changes in and Disagreements with Accountants on Accounting and Financial Disclosure:
Not applicable.
Item 9A. Controls and Procedures:
The company’s management evaluated, with the participation of the Chief Executive Officer and Chief Financ


## 2.4 Load the IBM 2025 Annual Report

IBM's 10-K incorporates the complete MD&A and Financial Statements by reference from the 2025 Annual Report.

The Annual Report will therefore be used to obtain these two sections for the RAG dataset.

In [8]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 2.3 MB/s eta 0:00:00


In [13]:
from pypdf import PdfReader

ANNUAL_REPORT_PATH = "/content/drive/MyDrive/Data Handling and Machine Learning/RAG/IBM/IBM_2025_Annual_Report.pdf"

reader = PdfReader(ANNUAL_REPORT_PATH)

print("Total PDF pages:", len(reader.pages))

Total PDF pages: 124


In [21]:
for i in range(min(12, len(reader.pages))):
    text = reader.pages[i].extract_text() or ""

    print(f"\n{'=' * 60}")
    print(f"PDF page {i + 1}")
    print(text[:500])


PDF page 1
IBM 2025 Annual Report I
2025 Annual Report
Let’s Create  
Smarter Business

PDF page 2
II IBM 2025 Annual Report 1
Dear IBM Investor:
IBM’s strategy is built on two technology 
foundations: hybrid cloud and AI. Together, 
they unlock the full value of enterprise 
data and enable businesses to operate 
with greater intelligence, agility, and scale. 
We have repositioned IBM as a software-
led, fully integrated platform company—
combining technology, consulting, and 
infrastructure. And we’re investing in the 
platforms that will define the next era of 
computing, including quantum.

PDF page 3
II IBM 2025 Annual Report 1
Dear IBM Investor:
IBM’s strategy is built on two technology 
foundations: hybrid cloud and AI. Together, 
they unlock the full value of enterprise 
data and enable businesses to operate 
with greater intelligence, agility, and scale. 
We have repositioned IBM as a software-
led, fully integrated platform company—
combining technology, consulting, and 
infr

## 2.5 Locate the Annual Report Section Boundaries

The Annual Report contains the complete Management Discussion and Financial Statements referenced by IBM's 10-K.

Because some PDF pages appear duplicated, I will identify the relevant section boundaries from the document content before extracting the final text.

In [22]:
# Search for likely Financial Statements starting pages

search_terms = [
    "CONSOLIDATED FINANCIAL STATEMENTS",
    "CONSOLIDATED INCOME STATEMENT",
    "CONSOLIDATED STATEMENT OF EARNINGS",
    "CONSOLIDATED BALANCE SHEET"
]

for term in search_terms:
    print(f"\n===== {term} =====")

    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""

        if term.lower() in text.lower():
            print(f"PDF page {i + 1}")
            print(text[:300])


===== CONSOLIDATED FINANCIAL STATEMENTS =====
PDF page 6
MANAGEMENT DISCUSSION NOTES TO CONSOLIDATED FINANCIAL STATEMENTS
Overview 6 Basis & Policies
Forward-Looking and Cautionary Statements 7 A Significant Accounting Policies 48
Management Discussion Snapshot 8 B Accounting Changes 62
Description of Business 11 Performance & Operations
Year in Review 16
PDF page 7
MANAGEMENT DISCUSSION NOTES TO CONSOLIDATED FINANCIAL STATEMENTS
Overview 6 Basis & Policies
Forward-Looking and Cautionary Statements 7 A Significant Accounting Policies 48
Management Discussion Snapshot 8 B Accounting Changes 62
Description of Business 11 Performance & Operations
Year in Review 16
PDF page 8
OVERVIEW
The financial section of the International Business Machines Corporation (IBM or “the company”) 2025 Annual Report includes the 
Management Discussion, the Consolidated Financial Statements and the Notes to Consolidated Financial Statements. This Overview 
is designed to provide the reader 
PDF page 9
OVERVI

# 3. Extract the RAG Sections

The IBM RAG dataset will use four sections:

- Business
- Risk Factors
- Management Discussion and Analysis (MD&A)
- Financial Statements and related notes

Business and Risk Factors are extracted from the 10-K filing.

The complete MD&A and Financial Statements are incorporated by reference in IBM's 10-K, so these sections are extracted from IBM's 2025 Annual Report.

Duplicate PDF pages are removed before processing.

In [14]:
# Extract Annual Report pages and remove consecutive duplicates

annual_pages = []
previous_text = None

for i, page in enumerate(reader.pages):
    text = page.extract_text() or ""
    text = text.strip()

    # Normalize whitespace for duplicate comparison
    normalized = " ".join(text.split())

    if normalized and normalized != previous_text:
        annual_pages.append({
            "pdf_page": i + 1,
            "text": text
        })

    previous_text = normalized

print("Original PDF pages:", len(reader.pages))
print("Unique pages:", len(annual_pages))

Original PDF pages: 124
Unique pages: 62


In [24]:
annual_text = "\n\n".join(page["text"] for page in annual_pages)

print("Annual Report characters:", len(annual_text))

Annual Report characters: 431582


In [25]:
business_text = ibm_clean[
    section_positions["Business"]:
    section_positions["Risk Factors"]
]

risk_text = ibm_clean[
    section_positions["Risk Factors"]:
    section_positions["MD&A"]
]

print("Business characters:", len(business_text))
print("Risk Factors characters:", len(risk_text))

Business characters: 10118
Risk Factors characters: 47045


## 3.1 Extract the Final RAG Sections

The Business and Risk Factors sections are extracted from IBM's 10-K.

The MD&A and Financial Statements are extracted from IBM's 2025 Annual Report using the page ranges referenced by the 10-K.

In [15]:
# Locate the actual Item 1B heading
item_1b_positions = []

start = 0

while True:
    position = ibm_clean.find("Item 1B. Unresolved Staff Comments", start)

    if position == -1:
        break

    item_1b_positions.append(position)
    start = position + 1

risk_end = item_1b_positions[-1]

business_text = ibm_clean[
    section_positions["Business"]:
    section_positions["Risk Factors"]
]

risk_text = ibm_clean[
    section_positions["Risk Factors"]:
    risk_end
]

print("Business characters:", len(business_text))
print("Risk Factors characters:", len(risk_text))

Business characters: 10118
Risk Factors characters: 36250


In [16]:
# MD&A: printed pages 6–38 -> PDF pages 8–40
mdna_text = "\n\n".join(
    page["text"]
    for page in annual_pages
    if 8 <= page["pdf_page"] <= 40
)

# Financial Statements + Notes:
# printed pages 42–116 -> PDF pages 44–118
financial_text = "\n\n".join(
    page["text"]
    for page in annual_pages
    if 44 <= page["pdf_page"] <= 118
)

print("MD&A characters:", len(mdna_text))
print("Financial Statements characters:", len(financial_text))

MD&A characters: 138934
Financial Statements characters: 266182


In [28]:
print("===== BUSINESS =====")
print(business_text[:300])

print("\n===== RISK FACTORS =====")
print(risk_text[:300])

print("\n===== MD&A =====")
print(mdna_text[:300])

print("\n===== FINANCIAL STATEMENTS =====")
print(financial_text[:300])

===== BUSINESS =====
Item 1. Business:
International Business Machines Corporation (IBM or the company) was incorporated in the State of New York on June 16, 1911, as the Computing-Tabulating-Recording Co. (C-T-R), a consolidation of the Computing Scale Co. of America, the Tabulating Machine Co. and The International Ti

===== RISK FACTORS =====
Item 1A. Risk Factors:
Risks Related to Our Business
Downturn in Economic Environment and Client Spending Budgets Could Impact the Company’s Business:
If overall demand for IBM’s products and solutions decreases, whether due to general economic conditions, or a shift in client buying patterns, the c

===== MD&A =====
OVERVIEW
The financial section of the International Business Machines Corporation (IBM or “the company”) 2025 Annual Report includes the 
Management Discussion, the Consolidated Financial Statements and the Notes to Consolidated Financial Statements. This Overview 
is designed to provide the reader 

===== FINANCIAL STATEMENTS ====

# 4. Chunk the RAG Sections

The extracted sections are divided into smaller text chunks for retrieval.

To keep the RAG pipeline consistent with the team baseline, I will use:

- Chunk size: 3,000 characters
- Chunk overlap: 300 characters
- Recursive character splitting

The overlap helps preserve context between neighboring chunks.

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", " ", ""]
)

section_texts = {
    "Business": business_text,
    "Risk Factors": risk_text,
    "MD&A": mdna_text,
    "Financial Statements": financial_text
}

section_chunks = {}

for section, text in section_texts.items():
    chunks = text_splitter.split_text(text)
    section_chunks[section] = chunks

    print(f"{section}: {len(chunks)} chunks")

Business: 4 chunks
Risk Factors: 16 chunks
MD&A: 60 chunks
Financial Statements: 116 chunks


In [30]:
for section, chunks in section_chunks.items():
    print(f"\n===== {section} =====")
    print("Number of chunks:", len(chunks))
    print("First chunk length:", len(chunks[0]))
    print("First chunk preview:")
    print(chunks[0][:300])


===== Business =====
Number of chunks: 4
First chunk length: 2989
First chunk preview:
Item 1. Business:
International Business Machines Corporation (IBM or the company) was incorporated in the State of New York on June 16, 1911, as the Computing-Tabulating-Recording Co. (C-T-R), a consolidation of the Computing Scale Co. of America, the Tabulating Machine Co. and The International Ti

===== Risk Factors =====
Number of chunks: 16
First chunk length: 2922
First chunk preview:
Item 1A. Risk Factors:
Risks Related to Our Business
Downturn in Economic Environment and Client Spending Budgets Could Impact the Company’s Business:
If overall demand for IBM’s products and solutions decreases, whether due to general economic conditions, or a shift in client buying patterns, the c

===== MD&A =====
Number of chunks: 60
First chunk length: 2958
First chunk preview:
OVERVIEW
The financial section of the International Business Machines Corporation (IBM or “the company”) 2025 Annual Report includes

# 5. Add Metadata to the Chunks

Each chunk is stored together with standardized metadata so that the documents from different companies can be combined later.

The metadata includes:

- company
- ticker
- filing_type
- filing_year
- section
- chunk_id
- source

In [22]:
chunk_records = []

source_map = {
    "Business": "IBM 2025 10-K",
    "Risk Factors": "IBM 2025 10-K",
    "MD&A": "IBM 2025 Annual Report",
    "Financial Statements": "IBM 2025 Annual Report"
}

for section, chunks in section_chunks.items():

    section_id = section.replace(" ", "_").replace("&", "and")

    for i, chunk in enumerate(chunks, start=1):

        chunk_id = f"IBM_{section_id}_{i:03d}"

        record = {
            "id": chunk_id,
            "text": chunk,
            "metadata": {
                "company": "International Business Machines Corporation",
                "ticker": "IBM",
                "filing_type": "10-K",
                "filing_year": 2025,
                "section": section,
                "chunk_id": chunk_id,
                "source": source_map[section]
            }
        }

        chunk_records.append(record)

print("Total chunk records:", len(chunk_records))

Total chunk records: 196


In [32]:
for record in chunk_records[:3]:
    print(record["metadata"])
    print(record["text"][:200])
    print("-" * 80)

{'company': 'International Business Machines Corporation', 'ticker': 'IBM', 'filing_type': '10-K', 'filing_year': 2025, 'section': 'Business', 'chunk_id': 'IBM_Business_001', 'source': 'IBM 2025 10-K'}
Item 1. Business:
International Business Machines Corporation (IBM or the company) was incorporated in the State of New York on June 16, 1911, as the Computing-Tabulating-Recording Co. (C-T-R), a cons
--------------------------------------------------------------------------------
{'company': 'International Business Machines Corporation', 'ticker': 'IBM', 'filing_type': '10-K', 'filing_year': 2025, 'section': 'Business', 'chunk_id': 'IBM_Business_002', 'source': 'IBM 2025 10-K'}
Our breadth and depth of expertise enables us to take different technologies and bring them together to solve the most pressing business issues of our clients. We differentiate from other providers by
--------------------------------------------------------------------------------
{'company': 'International Busin

# 6. Generate Embeddings

The RAG chunks are converted into numerical embeddings using the
`all-MiniLM-L6-v2` sentence-transformer model.

This is the same embedding model used for the common RAG baseline.

Each chunk will produce a 384-dimensional embedding that represents its semantic meaning.

In [33]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [34]:
sample_text = chunk_records[0]["text"]

sample_embedding = embedding_model.encode(sample_text)

print("Embedding type:", type(sample_embedding))
print("Embedding dimensions:", len(sample_embedding))
print("First 10 values:", sample_embedding[:10])

Embedding type: <class 'numpy.ndarray'>
Embedding dimensions: 384
First 10 values: [-0.06178225 -0.03797822 -0.02691194 -0.06844692 -0.03989004  0.00116881
  0.02090193  0.03532645  0.04210755 -0.03216363]


In [35]:
all_texts = [record["text"] for record in chunk_records]

all_embeddings = embedding_model.encode(
    all_texts,
    batch_size=16,
    show_progress_bar=True
)

print("Embeddings shape:", all_embeddings.shape)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Embeddings shape: (196, 384)


In [36]:
import numpy as np

print("Number of records:", len(chunk_records))
print("Number of embeddings:", len(all_embeddings))
print("Embedding dimensions:", all_embeddings.shape[1])
print("Contains NaN:", np.isnan(all_embeddings).any())
print("Contains infinity:", np.isinf(all_embeddings).any())

assert len(chunk_records) == len(all_embeddings)

print("\nEmbedding validation complete.")

Number of records: 196
Number of embeddings: 196
Embedding dimensions: 384
Contains NaN: False
Contains infinity: False

Embedding validation complete.


# 7. Store the RAG Data in ChromaDB

ChromaDB will store the IBM chunk text, embeddings, and metadata so that the most relevant chunks can be retrieved based on semantic similarity.

A separate local collection is used for IBM testing before the processed data is integrated into the shared team knowledge base.

In [38]:
import chromadb

CHROMA_PATH = "/content/drive/MyDrive/Data Handling and Machine Learning/chroma_database_ibm"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name="ibm_rag"
)

print("Collection name:", collection.name)
print("Current records:", collection.count())

Collection name: ibm_rag
Current records: 0


## 7.1 Add IBM Chunks to ChromaDB

The processed IBM chunks, embeddings, and metadata are now added to the local ChromaDB collection.

This enables semantic similarity search across the four selected sections.

In [39]:
ids = [
    record["metadata"]["chunk_id"]
    for record in chunk_records
]

documents = [
    record["text"]
    for record in chunk_records
]

metadatas = [
    record["metadata"]
    for record in chunk_records
]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=all_embeddings.tolist(),
    metadatas=metadatas
)

print("Chunks stored successfully.")
print("Number of records:", collection.count())

Chunks stored successfully.
Number of records: 196


# 8. Test Semantic Retrieval

The IBM ChromaDB collection is tested using several questions related to the four RAG sections.

Each question is converted into an embedding and compared with the stored chunk embeddings.

For ChromaDB distance scores, a smaller distance indicates a closer semantic match.

In [68]:
def retrieve_ibm(query, n_results=5, section=None):
    query_embedding = embedding_model.encode(query).tolist()

    query_args = {
        "query_embeddings": [query_embedding],
        "n_results": n_results
    }

    if section is not None:
        query_args["where"] = {"section": section}

    results = collection.query(**query_args)

    retrieved_chunks = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved_chunks.append({
            "text": document,
            "metadata": metadata,
            "distance": distance
        })

    return retrieved_chunks

print("IBM retrieval function ready.")

IBM retrieval function ready.


## 8.1 Retrieval Test — IBM Business Information

Question:

"What are IBM's main business areas?"

Retrieve the top 5 IBM chunks and inspect whether the retrieved evidence is relevant to the question.

In [41]:
question = "What are IBM's main business areas?"

retrieved = retrieve_ibm(
    question,
    n_results=5
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

Rank 1
Distance: 0.8370
Section: Risk Factors
Chunk ID: IBM_Risk_Factors_001

Item 1A. Risk Factors:
Risks Related to Our Business
Downturn in Economic Environment and Client Spending Budgets Could Impact the Company’s Business:
If overall demand for IBM’s products and solutions decreases, whether due to general economic conditions, or a shift in client buying patterns, the company’s revenue and profit could be impacted.
Failure of Innovation Initiatives Could Impact the Long-Term Success of the Company:
IBM has moved into areas, including those that incorporate or utilize hybrid cloud, AI, quantum and other disruptive technologies, in which it can differentiate itself through responsible innovation, by leveraging its investments in R&D and attracting a successful developer ecosystem. If IBM is unable to continue its cutting-edge innovation in a highly competitive and rapidly evolving environment or is unable to commercialize such innovations, expand and scale them with sufficient spee

### Observation

Relevant Business information was retrieved within the top 5 results.

The most relevant Business chunk appeared at Rank 3. Some Risk Factors chunks ranked higher because they also contain general descriptions of IBM's products, technologies, and business activities.

In [42]:
question = "What products, services, and business segments does IBM operate in?"

retrieved = retrieve_ibm(
    question,
    n_results=5
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

Rank 1
Distance: 0.7331
Section: Business
Chunk ID: IBM_Business_001

Item 1. Business:
International Business Machines Corporation (IBM or the company) was incorporated in the State of New York on June 16, 1911, as the Computing-Tabulating-Recording Co. (C-T-R), a consolidation of the Computing Scale Co. of America, the Tabulating Machine Co. and The International Time Recording Co. of New York. Since that time, IBM has focused on the intersection of business insight and technological innovation, and its operations and aims have been international in nature. This was signaled over 100 years ago, in 1924, when C-T-R changed its name to International Business Machines Corporation. And it continues today—we create sustained value for clients by helping them leverage the power of hybrid cloud and artificial intelligence (AI). Our hybrid cloud platform and AI technology support clients’ digital transformations and helps them reimagine critical workflows, at scale, and modernize application

### Observation

The more specific business question improved retrieval quality.

The relevant Business chunk was retrieved at Rank 1, showing that the embedding model performs better when the query includes more specific terms such as products, services, and business segments.

## 8.2 Retrieval Test — IBM Risk Factors

Question:

"What major risks could negatively affect IBM's business?"

Retrieve the top 5 IBM chunks and inspect whether Risk Factors content is ranked highly.

In [43]:
question = "What major risks could negatively affect IBM's business?"

retrieved = retrieve_ibm(
    question,
    n_results=5
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

Rank 1
Distance: 0.4727
Section: Risk Factors
Chunk ID: IBM_Risk_Factors_001

Item 1A. Risk Factors:
Risks Related to Our Business
Downturn in Economic Environment and Client Spending Budgets Could Impact the Company’s Business:
If overall demand for IBM’s products and solutions decreases, whether due to general economic conditions, or a shift in client buying patterns, the company’s revenue and profit could be impacted.
Failure of Innovation Initiatives Could Impact the Long-Term Success of the Company:
IBM has moved into areas, including those that incorporate or utilize hybrid cloud, AI, quantum and other disruptive technologies, in which it can differentiate itself through responsible innovation, by leveraging its investments in R&D and attracting a successful developer ecosystem. If IBM is unable to continue its cutting-edge innovation in a highly competitive and rapidly evolving environment or is unable to commercialize such innovations, expand and scale them with sufficient spee

### Observation

The Risk Factors retrieval performed well.

All top 5 retrieved chunks came from the Risk Factors section, with the most relevant chunk ranked first. This shows that the embedding model can clearly distinguish risk-related queries from the other sections.

## 8.3 Retrieval Test — IBM MD&A

Question:

"How did IBM perform financially in 2025 compared with 2024?"

Retrieve the top 5 IBM chunks and inspect whether MD&A content is ranked highly.

In [44]:
question = "How did IBM perform financially in 2025 compared with 2024?"

retrieved = retrieve_ibm(
    question,
    n_results=5
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

Rank 1
Distance: 0.7620
Section: Risk Factors
Chunk ID: IBM_Risk_Factors_004

The Company’s Financial Results for Particular Periods Are Difficult to Predict:
IBM’s revenues and profitability are affected by such factors as the introduction of new products and services, the ability to compete effectively in increasingly competitive marketplaces, the length of the sales cycles and the seasonality of technology purchases. In addition, certain of the company’s growth areas involve new products, new customers, new and evolving competitors, and new markets, all of which contribute to the difficulty of predicting the company’s financial results. The company’s financial results may also be impacted by the structure of products and services contracts and the nature of its customers’ businesses; for example, certain of the company’s services contracts with commercial customers in regulated industries are subject to periodic review by regulators with respect to controls and processes. Further, g

### Observation

Relevant MD&A information was retrieved within the top 5 results, with an MD&A chunk appearing at Rank 2.

However, the top-ranked result came from the Risk Factors section because the question was broad and contained financial-performance terms that also appear in risk-related content.

## 8.4 Retrieval Test — IBM Financial Statements

Question:

"What were IBM's total revenue and pre-tax income in 2025?"

Retrieve the top 5 IBM chunks and inspect whether Financial Statements content is ranked highly.

In [45]:
question = "What were IBM's total revenue and pre-tax income in 2025?"

retrieved = retrieve_ibm(
    question,
    n_results=5
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

Rank 1
Distance: 0.5638
Section: Financial Statements
Chunk ID: IBM_Financial_Statements_007

NOTE A. SIGNIFICANT ACCOUNTING POLICIES
Basis of Presentation
The accompanying Consolidated Financial Statements and footnotes of the International Business Machines Corporation (IBM or 
“the company”) have been prepared in accordance with accounting principles generally accepted in the United States of America 
(GAAP).
Within the financial statements and tables presented, certain columns and rows may not add due to the use of rounded numbers for 
disclosure purposes. Percentages presented are calculated from the underlying whole-dollar amounts. Certain prior-year amounts 
have been reclassified to conform to the change in current year presentation. This is annotated where applicable.
In the first quarter of 2025, the company made changes to the reported revenue categories within its Software and Consulting 
reportable segments. These changes did not impact the company's Consolidated Financial

### Observation

The Financial Statements retrieval performed well.

Financial Statements chunks occupied four of the top 5 positions. The retrieved evidence included IBM's 2025 total revenue and pre-tax income, showing that the semantic retrieval can locate relevant quantitative financial information.

# 9. Baseline Retrieval Summary

The semantic retrieval tests were performed using representative questions from the four selected RAG sections.

The results show that relevant evidence was retrieved within the top 5 results for all four question types.

- **Business:** Relevant Business content was retrieved at Rank 1 for the more specific business query.
- **Risk Factors:** Risk Factors content occupied all top 5 positions, with the most relevant chunk ranked first.
- **MD&A:** Relevant MD&A content appeared within the top 5, although a Risk Factors chunk ranked first for the broad financial-performance question.
- **Financial Statements:** Financial Statements content occupied four of the top 5 positions and included the requested financial figures.

Overall, the baseline semantic retrieval is able to retrieve relevant IBM information across the selected 10-K sections.

# 10. Generate RAG Answers with Fine-Tuned Qwen

The retrieval pipeline identifies relevant IBM source chunks from ChromaDB.

The retrieved evidence will now be provided to the fine-tuned InvestIQ Qwen model together with the user's question.

The model will use the retrieved context to generate a grounded answer and identify the source chunks supporting the response.

## 10.1 Load the Fine-Tuned Qwen Model

The retrieved IBM chunks will be used as context for the fine-tuned InvestIQ Qwen model.

The base Qwen model is loaded first, followed by the saved LoRA adapter.

In [46]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [47]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [51]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_PATH = "/content/drive/MyDrive/Data Handling and Machine Learning/week_04/outputs/Finetuned_version01/InvestIQ_finetuned_adapter_updated"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

trained_model.eval()

print("Fine-tuned InvestIQ Qwen model loaded successfully.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Fine-tuned InvestIQ Qwen model loaded successfully.


## 10.2 Build the RAG Context

The user's question is first passed through the IBM retrieval pipeline.

The top relevant chunks are then combined into a context block that will be provided to the fine-tuned Qwen model.

In [65]:
def build_ibm_context(retrieved_chunks):
    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"[Retrieved Source {i}]\n"
            f"Company: {chunk['metadata']['company']}\n"
            f"Section: {chunk['metadata']['section']}\n"
            f"Chunk ID: {chunk['metadata']['chunk_id']}\n"
            f"Text:\n{chunk['text']}"
        )

    return "\n\n".join(context_parts)

## 10.3 Generate Grounded RAG Answers

The retrieved IBM evidence is provided to the fine-tuned Qwen model together with the user's question.

The model is instructed to answer only from the retrieved context and to reference the exact chunk IDs supporting the response.

In [62]:
def generate_qwen_response(prompt, max_new_tokens=400):

    messages = [
        {"role": "user", "content": prompt}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(trained_model.device)

    with torch.no_grad():
        outputs = trained_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [71]:
def answer_with_ibm_rag(question, n_results=3, section=None):

    # Step 1: Retrieve relevant IBM chunks
    retrieved_chunks = retrieve_ibm(
        question,
        n_results=n_results,
        section=section
    )

    # Step 2: Build retrieved context
    retrieved_context = build_ibm_context(retrieved_chunks)

    # Step 3: Build grounded prompt
    rag_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Instructions:
- Read the entire retrieved context carefully.
- Answer EVERY part of the user's question.
- When the question asks for multiple financial values, extract every requested value.
- Preserve the units stated in the source.
- Do not omit a requested value if it is explicitly present in the context.
- Do not use general knowledge.
- Do not invent information.
- Cite only the chunk IDs that contain the information used.

Return ONLY valid JSON using this structure:

{{
  "answer": "A complete answer to every part of the question.",
  "sources": ["chunk_id"]
}}

User question:
{question}

Retrieved context:
{retrieved_context}
"""

    response = generate_qwen_response(rag_prompt)

    return {
        "response": response,
        "retrieved_chunks": retrieved_chunks
    }

In [74]:
result = answer_with_ibm_rag(
    question="What were IBM's total revenue and pre-tax income from continuing operations in 2025?",
    n_results=2,
    section="Financial Statements"
)

print("QWEN RESPONSE:")
print(result["response"])

print("\nRETRIEVED SOURCES:")
for chunk in result["retrieved_chunks"]:
    print(
        chunk["metadata"]["chunk_id"],
        "-",
        chunk["metadata"]["section"]
    )

QWEN RESPONSE:
{"answer":"The total revenue for 2025 was $67,535 million and the pre-tax income from continuing operations was $10,328 million.","sources":["IBM_Financial_Statements_046","IBM_Financial_Statements_007"]}]

RETRIEVED SOURCES:
IBM_Financial_Statements_046 - Financial Statements
IBM_Financial_Statements_007 - Financial Statements


### Observation

The grounded RAG pipeline successfully retrieved the relevant IBM financial statement chunk and generated the correct answer.

For the 2025 financial question, the model identified total revenue of `$`67,535 million and pre-tax income from continuing operations of `$`10,328 million.

The most relevant supporting evidence was retrieved from `IBM_Financial_Statements_046`.

## 10.4 End-to-End RAG Testing

The completed IBM RAG pipeline is tested using representative questions from the Business, Risk Factors, MD&A, and Financial Statements sections.

Each question is restricted to its relevant section so that the retrieval stage returns focused evidence before the fine-tuned Qwen model generates the final grounded response.

In [77]:
test_questions = [
    (
        "Business",
        "How does IBM describe its main business activities and offerings?"
    ),
    (
        "Risk Factors",
        "What risks does IBM identify related to competition and technological change?"
    ),
    (
        "MD&A",
        "What factors influenced IBM's financial performance in 2025?"
    ),
    (
        "Financial Statements",
        "What was IBM's total revenue in 2025?"
    )
]

for section, question in test_questions:
    print("=" * 100)
    print("SECTION:", section)
    print("QUESTION:", question)

    result = answer_with_ibm_rag(
        question=question,
        n_results=3,
        section=section
    )

    print("\nQWEN RESPONSE:")
    print(result["response"])

    print("\nRETRIEVED SOURCES:")
    for chunk in result["retrieved_chunks"]:
        print(
            chunk["metadata"]["chunk_id"],
            "-",
            chunk["metadata"]["section"]
        )

    print()

SECTION: Business
QUESTION: How does IBM describe its main business activities and offerings?

QWEN RESPONSE:
{"answer":"The main business activities described by IBM include creating sustainable value for clients by leveraging hybrid cloud and AI technology, supporting digital transformations, and delivering business outcomes through software, consulting services, and hybrid cloud infrastructure.","sources":["IBM_Business_001","IBM_Business_004","IBM_Business_002"],"key_factors":["Hybrid cloud and AI technology","Digital transformations","Business outcome delivery","Software offerings","Consulting services","Hybrid cloud infrastructure"],"risks":["Competitive environment","Market volatility","New competitor exposure"],"probability_interpretation":"","recommendation_context":"The description indicates that IBM focuses on business insights combined with technological innovation rather than simply competing based on single metrics like price or size.","response_type":"financial_explanati

## 10.5 End-to-End Test Summary

The final IBM RAG pipeline was tested using representative questions from the Business, Risk Factors, MD&A, and Financial Statements sections.

In each test, retrieval was restricted to the relevant section before the retrieved evidence was passed to the fine-tuned Qwen model.

The results show that the pipeline was able to retrieve relevant IBM evidence and generate grounded responses across all four selected sections.

Overall, the end-to-end tests demonstrate that the IBM RAG pipeline can combine section-aware retrieval with the fine-tuned Qwen model to answer questions using the selected 10-K and Annual Report content.

# 11. Export IBM RAG Chunks and Metadata

The processed IBM chunks and their metadata are exported to JSON for later integration with the team RAG system.

In [23]:
import json

OUTPUT_PATH = "/content/drive/MyDrive/Data Handling and Machine Learning/week_05/outputs/dinal_ibm_processed_chunks.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        chunk_records,
        f,
        ensure_ascii=False,
        indent=2
    )

print("IBM RAG chunks exported successfully.")
print("Saved to:", OUTPUT_PATH)

IBM RAG chunks exported successfully.
Saved to: /content/drive/MyDrive/Data Handling and Machine Learning/week_05/outputs/dinal_ibm_processed_chunks.json
